In [ ]:
%load_ext autoreload
%autoreload 2

import carla
from autonomy.config import STABLE_TOLERANCE
from autonomy.carla_env import connect, hide_map_layers, spawn_vehicle, CameraRig, SpeedController
from autonomy.lane_follower import WaypointSteering
from autonomy.recorder import DataRecorder

In [ ]:
client, world = connect()
print(world)

In [3]:
print(world.get_map().name)

Carla/Maps/Town10HD_Opt


In [ ]:
# remove all the layers of the map except for the road itself
hide_map_layers(world)

In [ ]:
# spawn a car at a fixed recommended spawn point (same car + location every run)
# set spawn_index to ON_TURN_INDEX or ON_STRAIGHT_INDEX (both defined in autonomy/config.py)
# re-running would collide with the previous car at the same spot, so remove it first
from autonomy.config import ON_TURN_INDEX, ON_STRAIGHT_INDEX

spawn_index = ON_TURN_INDEX

try:
    vehicle.destroy()
except NameError:
    pass

vehicle = spawn_vehicle(world, spawn_index=spawn_index)

In [ ]:
# place forward facing camera attached to the ego vehicle
# re-running would leave the old camera attached, so remove it first
try:
    rig.destroy()
except NameError:
    pass

rig = CameraRig(world, vehicle)

In [ ]:
# visualize waypoints
# for wp in world.get_map().generate_waypoints(2.0):  # one every 2 m on all lanes
#     world.debug.draw_point(wp.transform.location, size=0.1, life_time=120)

In [ ]:
# start autopilot: waypoint steering + speed controller; recording starts once the speed is stable
steering = WaypointSteering(vehicle, world.get_map())
speed_controller = SpeedController(vehicle)
recorder = DataRecorder()  # wipes any previous recording
rig.clear()  # drop any frames captured before the run starts

recording = False

try:
    while True:
        snapshot = world.wait_for_tick()  # only update once per simulator tick

        throttle = speed_controller.step()
        steer = steering.steer()
        vehicle.apply_control(carla.VehicleControl(throttle=throttle, steer=steer))
        print(speed_controller.speed, steer)

        # start recording as soon as the speed is stable at the target (stays on afterwards)
        if not recording and speed_controller.is_stable(STABLE_TOLERANCE):
            recording = True
            print(f"Speed stable at {speed_controller.speed:.2f}, recording started")

        # camera frames are discarded until recording starts; then each is matched with the steer of its sim frame
        images = rig.drain()
        if recording:
            recorder.update(snapshot.frame, steer, images)
finally:
    recorder.close()